# 🚀 Machine Learning & Deep Learning Pipeline: Phát Hiện Văn Bản AI vs Human
---
Pipeline này thực hiện hoàn chỉnh quy trình so sánh đối đầu **3 Nhánh Mô Hình Đột Phá**:
- **Nhánh 1 (Strong Baseline)**: Dual-Channel TF-IDF (Word n-grams + Char n-grams) + Logistic Regression.
- **Nhánh 3 (Stylometry + Tree)**: Kết hợp Đặc trưng Phong cách Ngôn ngữ học (EDA) + TF-IDF + Gradient Boosting (XGBoost / HistGradientBoosting).
- **Nhánh 4 (Deep Learning)**: Mạng Nơ-ron Phân loại Sâu (Deep Neural Classifier) trên không gian kết hợp.

### Quy trình chuẩn mực:
1. **Data Loading & Preprocessing** (Làm sạch, khử trùng lặp).
2. **Stratified Split** (Train / Validation / Test - Tuyệt đối không rò rỉ dữ liệu).
3. **Thực thi 3 Nhánh độc lập**.
4. **Đánh giá đối đầu (Model Evaluation)**: Macro-F1, Precision/Recall, ROC-AUC, PR-AUC, Confusion Matrix.


In [ ]:
# ==============================================================================
# 1. CÀI ĐẶT & NẠP THƯ VIỆN
# ==============================================================================
import os
import re
import time
import string
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, 
    roc_auc_score, average_precision_score, 
    confusion_matrix, roc_curve, precision_recall_curve
)
from scipy.sparse import hstack, csr_matrix

# Kiểm tra thư viện Gradient Boosting
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

from sklearn.ensemble import HistGradientBoostingClassifier

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110

print("Đã nạp thành công các thư viện nền tảng!")
print(f"XGBoost sẵn sàng: {HAS_XGB} | LightGBM sẵn sàng: {HAS_LGB}")


## 1. Nạp Dữ Liệu & Làm Sạch (Data Loading & Cleaning)
- Tự động nhận diện đường dẫn `AI_Human.csv`.
- Lọc bỏ văn bản rỗng và các bản ghi bị trùng lặp (`drop_duplicates`).
- Thiết lập `SAMPLE_SIZE` linh hoạt (mặc định 50,000 dòng để chạy nhanh, có thể chỉnh `None` để chạy full 487k dòng).


In [ ]:
# ==============================================================================
# 2. NẠP DỮ LIỆU & LÀM SẠCH
# ==============================================================================
possible_paths = [
    os.path.join('data', 'AI_Human.csv'),
    'AI_Human.csv',
    os.path.join('..', 'data', 'AI_Human.csv')
]
DATA_PATH = next((p for p in possible_paths if os.path.exists(p)), 'AI_Human.csv')

SAMPLE_SIZE = 50000  # Đặt None nếu muốn chạy toàn bộ 487,235 dòng
RANDOM_STATE = 42

print(f"Đang nạp dữ liệu từ: {DATA_PATH} ...")
t0 = time.time()
df_raw = pd.read_csv(DATA_PATH)
print(f"Đã nạp {len(df_raw):,} dòng gốc trong {time.time() - t0:.2f}s.")

# --- BƯỚC LÀM SẠCH (DATA CLEANING) ---
# 1. Xóa giá trị null
df_clean = df_raw.dropna(subset=['text', 'generated']).copy()

# 2. Xóa các dòng văn bản trống hoặc chỉ chứa khoảng trắng
df_clean = df_clean[df_clean['text'].str.strip().str.len() > 10].copy()

# 3. Xóa các bài viết trùng lặp (Duplicates)
dups_count = df_clean['text'].duplicated().sum()
df_clean = df_clean.drop_duplicates(subset=['text']).reset_index(drop=True)

# 4. Chuẩn hóa nhãn thành kiểu int (0: Human, 1: AI)
df_clean['target'] = df_clean['generated'].astype(int)

print(f"Đã xóa {dups_count:,} dòng trùng lặp. Còn lại {len(df_clean):,} mẫu sạch.")

# Lấy mẫu phân tầng (nếu cấu hình SAMPLE_SIZE)
if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(df_clean):
    df_sample, _ = train_test_split(
        df_clean,
        train_size=SAMPLE_SIZE,
        stratify=df_clean['target'],
        random_state=RANDOM_STATE
    )
    df = df_sample.copy().reset_index(drop=True)
    print(f"-> Đã lấy mẫu phân tầng {len(df):,} dòng để huấn luyện pipeline.")
else:
    df = df_clean.copy()
    print(f"-> Sử dụng toàn bộ dữ liệu sạch: {len(df):,} dòng.")

print(f"Tỷ lệ phân bố nhãn: Human (0) = {(df['target']==0).mean()*100:.1f}% | AI (1) = {(df['target']==1).mean()*100:.1f}%")


## 2. Phân Chia Tập Dữ Liệu (Stratified Train / Val / Test Split)
- **Train (70%)**: Dùng để huấn luyện mô hình và fit các bộ trích xuất đặc trưng (TF-IDF, Scaler).
- **Validation (15%)**: Dùng để tinh chỉnh ngưỡng phân loại và Early Stopping của Deep Learning/Tree.
- **Test (15%)**: Đánh giá độc lập cuối cùng — **Tuyệt đối không rò rỉ dữ liệu (No Data Leakage)**.


In [ ]:
# ==============================================================================
# 3. PHÂN CHIA DỮ LIỆU CHỐNG RÒ RỈ (NO DATA LEAKAGE)
# ==============================================================================
# Tách trước tập Test (15%)
train_val_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['target'], random_state=RANDOM_STATE
)

# Tách Train (70% tổng) và Val (15% tổng ~ 17.65% của train_val)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.1765, stratify=train_val_df['target'], random_state=RANDOM_STATE
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Tập Huấn luyện (Train):      {len(train_df):,} mẫu ({(len(train_df)/len(df))*100:.1f}%)")
print(f"Tập Kiểm định (Validation): {len(val_df):,} mẫu ({(len(val_df)/len(df))*100:.1f}%)")
print(f"Tập Đánh giá (Test):         {len(test_df):,} mẫu ({(len(test_df)/len(df))*100:.1f}%)")


## 3. Nhánh 1: Dual-Channel TF-IDF + Logistic Regression
### Nguyên lý:
- **Kênh 1 (Word N-grams: 1-2 words)**: Nắm bắt ngữ nghĩa cụm từ và từ vựng đặc trưng.
- **Kênh 2 (Char N-grams: 3-5 chars)**: Bắt phong cách cấu tạo hình thái từ, tiếp đầu ngữ/tiếp vĩ ngữ và thói quen chính tả (rất nhạy với phong cách máy viết).
- Bộ phân loại: **Logistic Regression** với `sublinear_tf=True` và chuẩn hóa L2.


In [ ]:
# ==============================================================================
# 4. NHÁNH 1: DUAL-CHANNEL TF-IDF + LOGISTIC REGRESSION
# ==============================================================================
print("Đang huấn luyện Nhánh 1: Dual-Channel TF-IDF + Logistic Regression...")
t_b1 = time.time()

# 1. Bộ trích xuất cấp độ Từ (Word-level)
word_vec = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=12000,
    sublinear_tf=True,
    stop_words='english'
)

# 2. Bộ trích xuất cấp độ Ký tự (Char-level)
char_vec = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=25000,
    sublinear_tf=True
)

# Ghép 2 kênh: CHỈ fit trên train_df
print("-> Đang fit vectorizers trên tập Train...")
X_train_word = word_vec.fit_transform(train_df['text'])
X_train_char = char_vec.fit_transform(train_df['text'])
X_train_b1 = hstack([X_train_word, X_train_char])

# Transform trên Val và Test
X_val_word = word_vec.transform(val_df['text'])
X_val_char = char_vec.transform(val_df['text'])
X_val_b1 = hstack([X_val_word, X_val_char])

X_test_word = word_vec.transform(test_df['text'])
X_test_char = char_vec.transform(test_df['text'])
X_test_b1 = hstack([X_test_word, X_test_char])

y_train = train_df['target'].values
y_val = val_df['target'].values
y_test = test_df['target'].values

# Huấn luyện Logistic Regression
model_b1 = LogisticRegression(C=2.0, max_iter=500, solver='lbfgs', random_state=RANDOM_STATE, n_jobs=-1)
model_b1.fit(X_train_b1, y_train)

# Dự đoán xác suất
prob_val_b1 = model_b1.predict_proba(X_val_b1)[:, 1]
prob_test_b1 = model_b1.predict_proba(X_test_b1)[:, 1]
pred_test_b1 = (prob_test_b1 >= 0.5).astype(int)

b1_duration = time.time() - t_b1
print(f"Nhánh 1 hoàn tất trong {b1_duration:.2f} giây.")


## 4. Nhánh 3: Đặc Trưng Phong Cách (Stylometry) + TF-IDF + Gradient Boosting
### Nguyên lý:
Văn bản AI và Human có sự khác biệt rõ rệt về phong cách viết. Nhánh này kết hợp:
1. **12 Đặc trưng Ngôn ngữ học & Phong cách (Handcrafted Stylometric Features)**:
   - Chiều dài từ trung bình, chiều dài câu trung bình.
   - **Độ biến thiên độ dài câu (Sentence Length Variance / Burstiness)**: Con người viết câu biến thiên mạnh, AI viết câu rất đều.
   - Độ đa dạng từ vựng (Type-Token Ratio / Lexical Diversity).
   - Mật độ các dấu câu chi tiết: phẩy, chấm, ngoặc kép, chấm hỏi, chấm than, gạch nối.
   - Tỷ lệ ký tự in hoa.
2. **Top 80 thành phần ngữ nghĩa tiềm ẩn từ TF-IDF** (sử dụng `TruncatedSVD` / LSA).
3. **Mô hình**: Gradient Boosting (`XGBoost` hoặc `HistGradientBoostingClassifier`).


In [ ]:
# ==============================================================================
# 5. NHÁNH 3: STYLOMETRY + SVD + GRADIENT BOOSTING
# ==============================================================================
print("Đang chuẩn bị đặc trưng cho Nhánh 3: Stylometry + Gradient Boosting...")
t_b3 = time.time()

def extract_stylometric_features(df_subset):
    features = pd.DataFrame(index=df_subset.index)
    
    # 1. Độ dài cơ bản
    features['char_count'] = df_subset['text'].str.len()
    words_series = df_subset['text'].str.split()
    features['word_count'] = words_series.apply(len).clip(lower=1)
    
    # 2. Phân tích câu & Độ biến thiên câu (Burstiness)
    def analyze_sentences(text):
        # Tách câu bằng dấu chấm, hỏi, than
        sents = [s.strip() for s in re.split(r'[.!?]+', text) if len(s.strip()) > 0]
        if not sents:
            return 1, len(text.split()), 0.0
        sent_lens = [len(s.split()) for s in sents]
        return len(sent_lens), np.mean(sent_lens), (np.std(sent_lens) if len(sent_lens) > 1 else 0.0)
    
    sent_stats = df_subset['text'].apply(analyze_sentences)
    features['sent_count'] = [s[0] for s in sent_stats]
    features['avg_sent_len'] = [s[1] for s in sent_stats]
    features['sent_len_std'] = [s[2] for s in sent_stats]  # Burstiness đặc trưng
    
    # 3. Độ dài từ trung bình & Độ phong phú từ vựng (TTR)
    features['avg_word_len'] = features['char_count'] / features['word_count']
    features['lexical_diversity'] = words_series.apply(lambda w: len(set(w)) / max(1, len(w)))
    
    # 4. Mật độ dấu câu (trên 100 từ)
    punct_set = set(string.punctuation)
    features['punct_count'] = df_subset['text'].apply(lambda x: sum(1 for c in x if c in punct_set))
    features['punct_density'] = (features['punct_count'] / features['word_count']) * 100
    
    features['comma_density'] = (df_subset['text'].apply(lambda x: x.count(',')) / features['word_count']) * 100
    features['period_density'] = (df_subset['text'].apply(lambda x: x.count('.')) / features['word_count']) * 100
    features['quote_density'] = (df_subset['text'].apply(lambda x: x.count('"') + x.count("'")) / features['word_count']) * 100
    features['uppercase_ratio'] = (df_subset['text'].apply(lambda x: sum(1 for c in x if c.isupper())) / features['char_count'].clip(lower=1)) * 100
    
    return features

# Trích xuất Stylometric features
print("-> Đang trích xuất đặc trưng phong cách...")
style_train = extract_stylometric_features(train_df)
style_val = extract_stylometric_features(val_df)
style_test = extract_stylometric_features(test_df)

# Chuẩn hóa (StandardScaler) - CHỈ fit trên train
scaler = StandardScaler()
X_style_train_scaled = scaler.fit_transform(style_train)
X_style_val_scaled = scaler.transform(style_val)
X_style_test_scaled = scaler.transform(style_test)

# Giảm chiều TF-IDF (LSA / SVD) lấy 80 components ngữ nghĩa
print("-> Đang giảm chiều TF-IDF (TruncatedSVD 80 components)...")
svd = TruncatedSVD(n_components=80, random_state=RANDOM_STATE)
X_svd_train = svd.fit_transform(X_train_b1)
X_svd_val = svd.transform(X_val_b1)
X_svd_test = svd.transform(X_test_b1)

# Ghép đặc trưng phong cách + đặc trưng SVD
X_train_b3 = np.hstack([X_style_train_scaled, X_svd_train])
X_val_b3 = np.hstack([X_style_val_scaled, X_svd_val])
X_test_b3 = np.hstack([X_style_test_scaled, X_svd_test])

print(f"Tổng số chiều đặc trưng Nhánh 3: {X_train_b3.shape[1]}")

# Huấn luyện Gradient Boosting (XGBoost hoặc HistGradientBoosting)
print("-> Đang huấn luyện mô hình Gradient Boosting...")
if HAS_XGB:
    model_b3 = xgb.XGBClassifier(
        n_estimators=300,
        learning_rate=0.08,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        eval_metric='logloss'
    )
    model_b3.fit(X_train_b3, y_train, eval_set=[(X_val_b3, y_val)], verbose=False)
else:
    model_b3 = HistGradientBoostingClassifier(
        max_iter=250,
        learning_rate=0.08,
        max_depth=8,
        random_state=RANDOM_STATE
    )
    model_b3.fit(X_train_b3, y_train)

prob_test_b3 = model_b3.predict_proba(X_test_b3)[:, 1]
pred_test_b3 = (prob_test_b3 >= 0.5).astype(int)

b3_duration = time.time() - t_b3
print(f"Nhánh 3 hoàn tất trong {b3_duration:.2f} giây.")


## 5. Nhánh 4: Deep Learning (Deep Neural Classifier)
### Nguyên lý:
- Huấn luyện một mạng nơ-ron sâu (**Deep Neural Network**) với các tầng `Linear`, `GELU activation`, `Dropout` và `AdamW optimizer`.
- Đầu vào là không gian biểu diễn đa chiều kết hợp giữa các đặc trưng phong cách ngôn ngữ và ngữ nghĩa LSA.
- Có cơ chế kiểm tra tính tương thích phần cứng an toàn, tự động chọn CPU/GPU tối ưu mà không lo lỗi kernel.


In [ ]:
# ==============================================================================
# 6. NHÁNH 4: DEEP LEARNING CLASSIFIER
# ==============================================================================
print("Đang khởi tạo Nhánh 4: Deep Learning Classifier...")
t_b4 = time.time()

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Kiểm tra an toàn thiết bị tính toán
def get_safe_device():
    if not torch.cuda.is_available():
        return torch.device('cpu'), "CPU (CUDA không khả dụng)"
    try:
        # Kiểm tra tính tương thích kernel thực tế
        bn = torch.nn.BatchNorm1d(2).cuda()
        x = torch.randn(2, 2).cuda()
        out = bn(x)
        out.sum().backward()
        return torch.device('cuda'), f"GPU ({torch.cuda.get_device_name(0)})"
    except Exception as e:
        return torch.device('cpu'), f"CPU (GPU {torch.cuda.get_device_name(0)} yêu cầu CUDA build tương thích hơn: {type(e).__name__})"

device, dev_msg = get_safe_device()
print(f"-> Thiết bị tính toán sử dụng: {dev_msg}")

# Kiến trúc Mạng Nơ-ron Phân loại Sâu
class DeepTextClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.layer_norm1 = nn.LayerNorm(hidden_dim)
        self.relu = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.layer_norm2 = nn.LayerNorm(hidden_dim // 2)
        self.out = nn.Linear(hidden_dim // 2, 1)
        
    def forward(self, x):
        x = self.drop(self.relu(self.layer_norm1(self.fc1(x))))
        x = self.drop(self.relu(self.layer_norm2(self.fc2(x))))
        return self.out(x).squeeze(-1)

# Chuyển dữ liệu sang PyTorch Tensors
X_tr_t = torch.tensor(X_train_b3, dtype=torch.float32)
y_tr_t = torch.tensor(y_train, dtype=torch.float32)
X_vl_t = torch.tensor(X_val_b3, dtype=torch.float32)
y_vl_t = torch.tensor(y_val, dtype=torch.float32)
X_ts_t = torch.tensor(X_test_b3, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=128, shuffle=True)
val_loader = DataLoader(TensorDataset(X_vl_t, y_vl_t), batch_size=256, shuffle=False)

model_b4 = DeepTextClassifier(input_dim=X_train_b3.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model_b4.parameters(), lr=1e-3, weight_decay=1e-4)

# Huấn luyện 12 epochs với Early Stopping
EPOCHS = 12
best_val_loss = float('inf')
best_weights = None

for epoch in range(1, EPOCHS + 1):
    model_b4.train()
    total_loss = 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model_b4(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(by)
        
    # Validation
    model_b4.eval()
    val_loss = 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            logits = model_b4(bx)
            val_loss += criterion(logits, by).item() * len(by)
            
    val_loss /= len(y_val)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = model_b4.state_dict()

# Nạp trọng số tốt nhất và dự đoán trên Test
model_b4.load_state_dict(best_weights)
model_b4.eval()
with torch.no_grad():
    test_logits = model_b4(X_ts_t.to(device))
    prob_test_b4 = torch.sigmoid(test_logits).cpu().numpy()
    pred_test_b4 = (prob_test_b4 >= 0.5).astype(int)

b4_duration = time.time() - t_b4
print(f"Nhánh 4 hoàn tất trong {b4_duration:.2f} giây.")


## 6. Đánh Giá Đối Đầu Giữa 3 Nhánh (Model Evaluation & Benchmarking)
So sánh trực tiếp cả 3 nhánh trên cùng một tập Test độc lập:
1. **Macro-F1 Score**: Trọng số cân bằng giữa 2 lớp (thước đo khách quan nhất).
2. **Per-class Precision & Recall**: Khả năng bắt đúng bài của Human và bài của AI.
3. **ROC-AUC & PR-AUC**: Khả năng phân biệt xác suất tổng quát.
4. **Confusion Matrix & Đường cong ROC**.


In [ ]:
# ==============================================================================
# 7. BẢNG TỔNG HỢP SO SÁNH HIỆU NĂNG TRÊN TẬP TEST
# ==============================================================================
models = {
    'Branch 1: Dual TF-IDF + Logistic Reg': (pred_test_b1, prob_test_b1, b1_duration),
    'Branch 3: Stylometry + SVD + GBDT': (pred_test_b3, prob_test_b3, b3_duration),
    'Branch 4: Deep Neural Classifier': (pred_test_b4, prob_test_b4, b4_duration)
}

results = []

for name, (pred, prob, duration) in models.items():
    acc = accuracy_score(y_test, pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, pred, average='macro')
    prec_ai, rec_ai, f1_ai, _ = precision_recall_fscore_support(y_test, pred, labels=[1], average=None)
    roc_auc = roc_auc_score(y_test, prob)
    pr_auc = average_precision_score(y_test, prob)
    
    results.append({
        'Mô Hình': name,
        'Accuracy': round(acc, 4),
        'Macro-F1': round(f1, 4),
        'Macro-Recall': round(rec, 4),
        'AI Precision': round(prec_ai[0], 4),
        'AI Recall': round(rec_ai[0], 4),
        'ROC-AUC': round(roc_auc, 4),
        'PR-AUC': round(pr_auc, 4),
        'Thời gian (s)': round(duration, 1)
    })

comparison_df = pd.DataFrame(results).sort_values(by='Macro-F1', ascending=False).reset_index(drop=True)
display(comparison_df)


In [ ]:
# ==============================================================================
# 8. TRỰC QUAN HÓA MA TRẬN NHẦM LẪN (CONFUSION MATRIX) VÀ ĐƯỜNG CONG ROC
# ==============================================================================
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

# 1. Confusion Matrix cho từng nhánh
branch_preds = [
    ('Branch 1: Dual TF-IDF', pred_test_b1),
    ('Branch 3: Stylometry + GBDT', pred_test_b3),
    ('Branch 4: Deep Classifier', pred_test_b4)
]

for idx, (title, pred) in enumerate(branch_preds):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', cbar=False,
        xticklabels=['Human (0)', 'AI (1)'],
        yticklabels=['Human (0)', 'AI (1)'],
        ax=axes[idx],
        annot_kws={'fontsize': 12, 'fontweight': 'bold'}
    )
    axes[idx].set_title(f'CM: {title}', fontweight='bold')
    axes[idx].set_xlabel('Dự đoán (Predicted)')
    axes[idx].set_ylabel('Thực tế (Actual)')

# 4. Đường cong ROC so sánh cả 3 nhánh
palette_models = ['#2b7bba', '#27ae60', '#e74c3c']
for (name, (_, prob, _)), color in zip(models.items(), palette_models):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    axes[3].plot(fpr, tpr, label=f'{name.split(":")[0]} (AUC={auc_val:.3f})', color=color, linewidth=2)

axes[3].plot([0, 1], [0, 1], 'k--', alpha=0.6)
axes[3].set_title('So Sánh Đường Cong ROC (ROC Curves)', fontweight='bold')
axes[3].set_xlabel('False Positive Rate (FPR)')
axes[3].set_ylabel('True Positive Rate (TPR)')
axes[3].legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

# In kết luận tóm tắt
top_model = comparison_df.iloc[0]
print(f"MÔ HÌNH DẪN ĐẦU: {top_model['Mô Hình']}")
print(f"- Macro-F1: {top_model['Macro-F1']:.4f} | ROC-AUC: {top_model['ROC-AUC']:.4f} | AI Recall: {top_model['AI Recall']:.4f}")
